# NLP HW2 — выбранное декодирование и greedy на одной T4

Загрузите kaggle_bundle.zip как Private Kaggle Dataset, подключите к notebook и включите GPU T4. Запустите с чистого kernel. Только первая GPU используется даже при выдаче двух T4. MODE="selected" выбирает локального победителя; MODE="greedy" возвращает зафиксированный baseline. Официальный public637/960 и T4 27.31/79.46s относятся к **прежнему greedy-запуску**, сообщённому пользователем, не к новому выбранному режиму.

Notebook не обучает, не исполняет generated-программы и не отправляет посылки. 64 токена — рекомендация бота; сохранён проверенный предел256. Для бота скачивается submission.jsonl; остальные файлы — для проверки.


In [ ]:
import os, sys, time
NOTEBOOK_STARTED = time.perf_counter()
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_initialized():
    raise RuntimeError('Restart the kernel; run this cell before initializing CUDA')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
from pathlib import Path
import json, subprocess, importlib.metadata
BUNDLE = None  # Optional explicit Path('/kaggle/input/your-dataset/kaggle_bundle')
MODE = 'selected'  # Change only this to 'greedy' for the fixed baseline
OFFICIAL_QUESTIONS = None  # Optional Path to pass1_questions.jsonl
OUTPUT = Path('/kaggle/working')
if BUNDLE is None:
    found, visited = [], set()
    for directory, subdirs, files in os.walk('/kaggle/input', followlinks=True):
        real = os.path.realpath(directory)
        if real in visited:
            subdirs[:] = []
            continue
        visited.add(real)
        if 'bundle_manifest.json' in files:
            path = Path(directory) / 'bundle_manifest.json'
            if json.loads(path.read_text()).get('bundle_kind') == 'hw2_kaggle_inference':
                found.append(path.parent.resolve())
        if OFFICIAL_QUESTIONS is None and 'pass1_questions.jsonl' in files:
            OFFICIAL_QUESTIONS = Path(directory) / 'pass1_questions.jsonl'
    if len(found) != 1:
        raise ValueError(f'Set BUNDLE explicitly; candidates: {found}')
    BUNDLE = found[0]
assert MODE in ['selected', 'greedy']
CONFIG_NAME = 'generation_selected.json' if MODE == 'selected' else 'generation_greedy_t4.json'
GENERATION_CONFIG = BUNDLE / 'configs' / CONFIG_NAME
print('Bundle:', BUNDLE, 'mode:', MODE)
print(GENERATION_CONFIG.read_text())


In [ ]:
install_started = time.perf_counter()
required = {'transformers':'4.46.3', 'tokenizers':'0.20.3'}
repair = False
for name, version in required.items():
    try:
        repair |= importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        repair = True
for name in ['numpy', 'safetensors']:
    try:
        importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        repair = True
if repair:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(BUNDLE/'requirements-inference.txt')], check=True)
DEPENDENCY_INSTALL_SECONDS = time.perf_counter() - install_started
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
print('Python:', sys.version)
for name in ['torch','transformers','tokenizers','numpy','safetensors']:
    print(name, importlib.metadata.version(name))
if 'T4' not in torch.cuda.get_device_name(0):
    print('Not a T4: do not claim the official T4 time bound from this run.')


Полный дочерний процесс измеряется отдельно: Python/imports, проверка файлов, загрузка, warmup, генерация и запись. В timing.json также раздельно записаны load, warmup и warmed generation; warmup не скрывается. Установка зависимостей измеряется отдельно. Для выбранного beam режима окончательный замер на T4 ещё необходим.


In [ ]:
OUTPUT.mkdir(exist_ok=True, parents=True)
started = time.perf_counter()
command = [sys.executable, str(BUNDLE/'generate.py'), '--bundle', str(BUNDLE),
           '--generation-config', str(GENERATION_CONFIG), '--output-dir', str(OUTPUT)]
with (OUTPUT/'inference.log').open('w') as log:
    result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
elapsed = time.perf_counter() - started
print((OUTPUT/'inference.log').read_text())
result.check_returncode()
timing = json.loads((OUTPUT/'timing.json').read_text())
timing['subprocess_end_to_end_seconds'] = elapsed
timing['dependency_install_seconds'] = DEPENDENCY_INSTALL_SECONDS
timing['mode'] = MODE
timing['one_T4_confirmed'] = 'T4' in torch.cuda.get_device_name(0)
timing['full_process_under_180_seconds'] = elapsed <= 180
timing['notebook_elapsed_including_setup_seconds'] = time.perf_counter() - NOTEBOOK_STARTED
(OUTPUT/'timing.json').write_text(json.dumps(timing, indent=2))
print(json.dumps(timing, indent=2))


In [ ]:
# Validate 1919 unchanged outputs; optional matching to official question order.
sys.path.insert(0, str(BUNDLE))
from src.submission import export_submission
validation = export_submission(BUNDLE, OUTPUT, OFFICIAL_QUESTIONS)
print(json.dumps(validation, indent=2))
if not validation['official_pass1_file_compared']:
    print('Official pass1_questions.jsonl is absent: exact order against that separate file is not verified. Bundled original test order is preserved.')
from IPython.display import display, FileLink
os.chdir(OUTPUT)
display(FileLink('submission.jsonl'))
display(FileLink('kaggle_results.zip'))
print('Download submission.jsonl for the bot. Return kaggle_results.zip and the executed notebook for review.')
